# 🎓 LectureAI — CV Worker (Colab GPU)

Bu notebook, Colab GPU'sunu kullanarak video görüntü analizini çalıştırır.

**Mimari:** Pub/Sub Pull Subscriber → `cv_engine.run()` → GCS upload → Orchestrator'a haber ver

**Kullanım:** Hücreleri sırasıyla çalıştır. Son hücre sonsuz dinleme moduna geçer.
Pipeline trigger'ı ayrıca çalıştırılır (`trigger_analysis.py`).

## 1️⃣ GPU Kontrol & Kurulum

In [ ]:
# GPU kontrolü
!nvidia-smi

# Repo'yu klonla (veya güncelle)
import os
REPO_DIR = "/content/LectureAI"

if os.path.exists(REPO_DIR):
    print("Repo mevcut, güncelleniyor...")
    !cd {REPO_DIR} && git pull origin main
else:
    print("Repo klonlanıyor...")
    !git clone https://github.com/pinaraltinok/LectureAI.git {REPO_DIR}

%cd {REPO_DIR}

# Bağımlılıkları kur
!pip install -q -r requirements.cv.txt
print("\n✅ Kurulum tamamlandı!")

## 2️⃣ Google Cloud Authentication (Tek Tık)

In [ ]:
from google.colab import auth
auth.authenticate_user()

import os
os.environ["GOOGLE_CLOUD_PROJECT"] = "senior-design-488908"

# Doğrulama
from google.cloud import storage
client = storage.Client()
buckets = [b.name for b in client.list_buckets() if "lectureai" in b.name]
print(f"✅ Auth başarılı! Erişilen bucket'lar: {buckets}")

## 3️⃣ CV Worker'ı Başlat (Sonsuz Dinleme)

Bu hücre çalıştırıldığında worker Pub/Sub'dan mesaj beklemeye başlar.
Başka bir yerden `trigger_analysis.py` ile iş gönderebilirsin.

**Durdurmak için:** Hücreyi durdur (⬛ butonu) veya `Ctrl+C`

In [ ]:
!python scripts/cv_colab_worker.py